## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt


device = 'cuda:7'

%load_ext autoreload
%autoreload 2

## Dataset

In [2]:
from datasets.MoG import MoG

n, d = 10000, 64                                    # < higher d, higher MI
true_rho = 0.7                                      # < higher rho, higher MI

dataset = MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.2, -0.1, 0, 0.3, 0.4], rhos=[-0.3, 0.5, 0.2, 0.4, 0.9])
X, Y = dataset.sample_data(n_samples = n)
MI = dataset.empirical_mutual_info()                # < MI by MC estimate

X, Y = X.to(device), Y.to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 6.889054910494017


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.lr = 5e-4
        self.bs = 500
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]


## MI estimate

In [4]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500


finished: t= 0 loss= 1.4803779125213623 loss val= 1.5074565410614014 best val loss= 1.5074565410614014 best t= 0


finished: t= 76 loss= 0.7407389879226685 loss val= 0.7594382762908936 best val loss= 0.7280677556991577 best t= 46


finished: t= 152 loss= 0.7523767352104187 loss val= 0.7674862146377563 best val loss= 0.7254774570465088 best t= 94


finished: t= 228 loss= 0.7584109306335449 loss val= 0.7709248661994934 best val loss= 0.7254774570465088 best t= 94


...

finished: t= 988 loss= 0.7278052568435669 loss val= 0.7228894233703613 best val loss= 0.7061780691146851 best t= 688


finished: t= 1064 loss= 0.7099202275276184 loss val= 0.7647949457168579 best val loss= 0.7061780691146851 best t= 688


finished: t= 1140 loss= 0.7296885251998901 loss val= 0.7281495928764343 best val loss= 0.7061780691146851 best t= 688


true MI: 6.888545351565563


est MI: 7.293962550163269


In [5]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.017414124682545662 loss val= 0.01861601322889328 best val loss= 0.01861601322889328 best t= 0


finished: t= 201 loss= -3.0458531379699707 loss val= -2.1821346282958984 best val loss= -2.4555485248565674 best t= 99


true MI: 6.901643786418516


est MI: 3.0949254035949707


In [5]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))


finished: t= 0 loss= 2.0507125854492188 loss val= 2.069305896759033 best val loss= 2.069305896759033 best t= 0
finished: t= 126 loss= 1.5689302682876587 loss val= 1.6604385375976562 best val loss= 1.6307024955749512 best t= 112
finished: t= 252 loss= 1.6279675960540771 loss val= 1.6627280712127686 best val loss= 1.6279056072235107 best t= 162


finished: t= 0 loss= 2.0559210777282715 loss val= 2.0756585597991943 best val loss= 2.0756585597991943 best t= 0
finished: t= 126 loss= 1.6388529539108276 loss val= 1.679556965827942 best val loss= 1.6390056610107422 best t= 38


finished: t= 0 loss= 567.150390625 loss val= 584.98291015625 best val loss= 584.98291015625 best t= 0
finished: t= 251 loss= 81.84658813476562 loss val= 86.76899719238281 best val loss= 86.76899719238281 best t= 251
finished: t= 502 loss= 80.94110107421875 loss val= 86.53428649902344 best val loss= 86.53428649902344 best t= 502
finished: t= 753 loss= 80.48675537109375 loss val= 86.52926635742188 best val loss= 86.529266

In [7]:
## MI estimate via flows
from estimators.MIENF import MIENF

estimator = MIENF(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))

MIENF (K=1, marginal_weight=0), joint learning True 

finished: t= 0 loss= 9322.6806640625 loss val= 9331.228515625 best val loss= 9331.228515625 best t= 0
finished: t= 101 loss= 94.1249008178711 loss val= 95.55921936035156 best val loss= 95.55921936035156 best t= 101
finished: t= 202 loss= 87.73372650146484 loss val= 92.70205688476562 best val loss= 92.09048461914062 best t= 159
finished: t= 303 loss= 86.50352478027344 loss val= 97.1007080078125 best val loss= 92.09048461914062 best t= 159


true MI: 6.900404047779712
est MI: 2.222219467163086
